# Week 1 — Logistics Data Analysis
## Strategic Planning & Data Exploration
**Dataset:** `business_data.csv` | **Columns:** Order_ID, Order_Date, Customer_ID, Customer_Type, Product, Category, Region, Quantity, Revenue, Profit, Profit_Category

---


## Step 1 — Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("business_data.csv")

print("Shape:", df.shape)
print("\nColumn dtypes:")
print(df.dtypes)
print("\nFirst 5 rows:")
df.head()


## Step 2 — Data Cleaning

In [ ]:
# Strip ₹ symbol and commas, convert Revenue and Profit to float
for col in ["Revenue", "Profit"]:
    df[col] = (df[col]
               .astype(str)
               .str.replace("₹", "", regex=False)
               .str.replace(",", "", regex=False)
               .astype(float))

# Parse Order_Date as datetime; extract Month and Quarter
df["Order_Date"] = pd.to_datetime(df["Order_Date"], dayfirst=True)
df["Month"]   = df["Order_Date"].dt.month
df["Quarter"] = df["Order_Date"].dt.quarter

# Handle anomalies
df["Customer_ID"] = df["Customer_ID"].replace("UNKNOWN", np.nan)
df["Region"]      = df["Region"].replace("Not Specified", np.nan)
df = df[df["Quantity"] > 0].reset_index(drop=True)

print(f"Clean dataset: {df.shape[0]} rows, {df.shape[1]} columns")
df.isnull().sum()


## Step 3 — KPI Computation

In [ ]:
# KPI 1: Profit Margin Rate (PMR)
df["Profit_Margin_Rate"] = (df["Profit"] / df["Revenue"]) * 100

pmr_region = df.groupby("Region")["Profit_Margin_Rate"].mean().sort_values(ascending=False)
print("KPI 1 — Profit Margin Rate by Region (%):")
print(pmr_region.round(2))

# KPI 2: Average Revenue per Order (ARPO) by Customer Type
arpo = df.groupby("Customer_Type")["Revenue"].mean()
print("\nKPI 2 — ARPO by Customer Type (₹):")
print(arpo.round(2))

# KPI 3: Order Volume Concentration Index (OVCI)
ovci = (df["Region"].value_counts() / len(df) * 100).round(2)
print("\nKPI 3 — OVCI by Region (%):")
print(ovci)


## Step 4 — Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution of Revenue
axes[0].hist(df["Revenue"], bins=30, color="#3b82d4", edgecolor="white")
axes[0].set_title("Revenue Distribution")
axes[0].set_xlabel("Revenue (₹)")

# Orders by Category
cat_counts = df["Category"].value_counts()
axes[1].bar(cat_counts.index, cat_counts.values, color="#7c5cd8")
axes[1].set_title("Orders by Category")
axes[1].tick_params(axis="x", rotation=30)

# Profit Margin Rate by Region
pmr_region.plot(kind="barh", ax=axes[2], color="#3b82d4")
axes[2].set_title("Avg PMR by Region (%)")
axes[2].set_xlabel("PMR (%)")

plt.tight_layout()
plt.savefig("eda_overview.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Heatmap: Profit Margin Rate — Region × Category
pivot = df.pivot_table(
    values="Profit_Margin_Rate",
    index="Region",
    columns="Category",
    aggfunc="mean"
)

plt.figure(figsize=(10, 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlGnBu",
            linewidths=0.5, linecolor="#e5e7eb")
plt.title("Avg Profit Margin Rate (%) — Region × Category", fontsize=13)
plt.tight_layout()
plt.savefig("heatmap_pmr.png", dpi=150, bbox_inches="tight")
plt.show()


## Step 5 — Feature Engineering

In [ ]:
from sklearn.preprocessing import LabelEncoder

df["Revenue_per_Unit"] = df["Revenue"] / df["Quantity"]

le = LabelEncoder()
for col in ["Category", "Region", "Customer_Type"]:
    df[col + "_enc"] = le.fit_transform(df[col].fillna("Unknown"))

print("New features added:")
print(df[["Revenue_per_Unit", "Category_enc", "Region_enc", "Customer_Type_enc"]].head())


## Step 6 — Baseline Regression Model (Predict Profit)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model  import LinearRegression
from sklearn.metrics       import mean_squared_error, r2_score

features = ["Quantity", "Revenue", "Category_enc",
            "Region_enc", "Customer_Type_enc", "Month"]
X = df[features]
y = df["Profit"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = mean_squared_error(y_test, y_pred, squared=False)
r2   = r2_score(y_test, y_pred)
print(f"Linear Regression — RMSE: ₹{rmse:,.2f}  |  R²: {r2:.4f}")


## Step 7 — K-Means Clustering (Regional Segmentation)

In [ ]:
from sklearn.cluster   import KMeans
from sklearn.metrics   import silhouette_score
from sklearn.preprocessing import StandardScaler

cluster_features = df.groupby("Region").agg(
    Avg_Revenue=("Revenue", "mean"),
    Avg_Profit=("Profit", "mean"),
    Total_Orders=("Order_ID", "count"),
    Avg_PMR=("Profit_Margin_Rate", "mean")
).reset_index()

X_cluster = StandardScaler().fit_transform(
    cluster_features[["Avg_Revenue", "Avg_Profit", "Total_Orders", "Avg_PMR"]])

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_features["Cluster"] = kmeans.fit_predict(X_cluster)

sil = silhouette_score(X_cluster, cluster_features["Cluster"])
print(f"Silhouette Score: {sil:.4f}\n")
print(cluster_features[["Region", "Cluster", "Avg_Revenue", "Avg_PMR"]].to_string(index=False))


## Step 8 — Classification Model (Predict Profit_Category)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics  import classification_report, accuracy_score

df["Target"] = (df["Profit Category"].str.strip() == "Profit").astype(int)

X_clf = df[features]
y_clf = df["Target"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_tr, y_tr)
y_pred_clf = clf.predict(X_te)

print(f"Accuracy: {accuracy_score(y_te, y_pred_clf):.4f}\n")
print(classification_report(y_te, y_pred_clf, target_names=["Loss","Profit"]))


---
## Summary of Results

| Model | Metric | Target |
|---|---|---|
| Linear Regression | R² | ≥ 0.80 |
| Random Forest Classifier | Accuracy | ≥ 85% |
| K-Means Clustering | Silhouette Score | > 0.40 |

**Next Steps (Week 2):** Deep EDA, advanced feature engineering, hyperparameter tuning with GridSearchCV.  
**Bonus (Week 4):** Streamlit dashboard deployment.
